# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset: {getattr(metadata, 'name', 'No name')}")
print(f"Description: {getattr(metadata, 'description', 'No description')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We inspect the available record sets and, for each, list the available fields and their `@id`s.

In [ ]:
# List all record sets with their @id from the Croissant metadata
record_sets = getattr(metadata, 'recordSet', [])

print('Available Record Sets:')
record_set_ids = []
for rs in record_sets:
    rs_id = rs.get('@id', None)
    name = rs.get('name', rs_id)
    print(f"- {name}: @id = {rs_id}")
    record_set_ids.append(rs_id)

# Show fields for each record set (referenced via @id as required)
for rs in record_sets:
    rs_id = rs.get('@id', None)
    print(f"\nFields for RecordSet {rs_id}:")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', None)
        field_name = field.get('name', field_id)
        print(f"  - {field_name}: @id = {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this notebook, we select the first available record set for demonstration. Adjust the `selected_record_set_id` and field IDs as needed for your own analysis.

In [ ]:
# If there are no record sets defined in metadata, try to find default set
if len(record_set_ids) == 0:
    # Try known IDs from common Croissant record set naming
    record_set_ids = ['cr:main']
    print('No recordSet listed in metadata; try default @id = cr:main')

# Pick the first record set for illustration, or set your record set @id explicitly
selected_record_set_id = record_set_ids[0]
print(f"\nExtracting records from RecordSet: {selected_record_set_id}")

# Extract all records into a DataFrame
records = list(ds.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)

print("Available columns (fields) in DataFrame:")
print(df.columns.tolist())

# Show the first few rows
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll choose a numeric field (e.g. peptide_count or MW if available) and group by another field such as protein accession or sample ID if present.

In [ ]:
# List columns to pick suitable numeric and grouping fields
print(f"Available columns: {df.columns.tolist()}")

# Example column selection, update the variables as per the real column names from df.columns
numeric_field = None
group_field = None

# Try to find a common numeric field for peptide count or MW
for col in df.columns:
    if ('peptide' in col.lower() and 'count' in col.lower()) or ('mw' == col.lower()):
        numeric_field = col
        break
if numeric_field is None:
    # Try first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

# Try to find a column for grouping (like accession or sample)
for col in df.columns:
    if ('accession' in col.lower() or 'sample' in col.lower() or 'group' in col.lower()) and col != numeric_field:
        group_field = col
        break

print(f"Using numeric_field={numeric_field} and group_field={group_field}.")

# Filter rows with numeric_field > threshold
threshold = 10
if numeric_field is not None:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Group by group_field if present
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check available columns and select a numeric field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we generate a histogram of the selected numeric field, and if a group field exists, a bar plot of means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric_field
if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Barplot if grouped_df exists
    if 'grouped_df' in locals() and group_field is not None and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        chart = sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        chart.set_xticklabels(chart.get_xticklabels(), rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 Mass Spectrometry dataset was successfully loaded using the `mlcroissant` library by schema `@id`.
- RecordSet and Field `@id`s allow precise referencing and extraction of records for analysis.
- Basic EDA highlights the structure and distribution of numeric properties such as peptide counts or molecular weights.
- This template can be customized for domain-specific analysis by targeting other fields, adjusting filters, or adding new visualizations.

### Next Steps:
- Refine field selection as per specific research questions.
- Explore additional record sets or relationships, if present.
- Integrate statistical tests or machine learning analyses as needed.